# Minecraft Survival Gym Quickstart

This notebook verifies the Gymnasium contract with the mock backend and can optionally connect to a real Minecraft client.

**Audience:** researchers and developers evaluating the environment before training an agent.

**Prerequisites:**

- repository dependencies installed with `uv sync --extra dev --extra notebook`;
- for the real backend, Minecraft must be running with the Fabric bridge;
- the player must already be inside a loaded single-player Survival world.

**Learning goals:**

1. inspect the action and observation spaces;
2. reset and step the deterministic mock environment;
3. inspect an RGB observation and structured state;
4. verify the Fabric bridge before connecting to the real game;
5. run a short, safe NOOP episode.


## Runtime layout for the real backend

The real test uses two terminals and the Minecraft window:

| Component | Responsibility |
|---|---|
| Terminal 1 | run `cd fabric && ./gradlew runClient` |
| Minecraft | open **Singleplayer**, enter a world, and wait for the HUD |
| Terminal 2 | launch Jupyter and execute this notebook |

The Minecraft title screen is not sufficient. Keep Terminal 1 and Minecraft open while the real-backend cells run.


## 1. Imports and deterministic setup

All state required by later cells is initialized here. The notebook defaults to the mock backend so it runs safely from top to bottom without Minecraft.


In [ ]:
from __future__ import annotations

import socket

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

import minecraft_gym
from minecraft_gym.actions import noop_action

SEED = 42
WIDTH = 128
HEIGHT = 128
FRAME_SKIP = 4

print(f"Environment ID: {minecraft_gym.ENV_ID}")


## 2. Validate the Gymnasium interface with the mock backend

The mock backend implements the same Python contract as the Fabric bridge, but it does not simulate Minecraft physics. It is useful for validating shapes, action encoding, wrappers, and training code.


In [ ]:
mock_env = gym.make(
    minecraft_gym.ENV_ID,
    backend="mock",
    width=WIDTH,
    height=HEIGHT,
    frame_skip=FRAME_SKIP,
    render_mode="rgb_array",
)

observation, reset_info = mock_env.reset(seed=SEED)

print("Action space:", mock_env.action_space)
print("Observation keys:", tuple(observation))
print("Reset info:", reset_info)
assert mock_env.observation_space.contains(observation)


### Inspect observation shapes

The policy receives a visual frame plus compact, structured state. Inventory arrays always contain 36 slots.


In [ ]:
observation_summary = {
    key: {
        "shape": tuple(np.asarray(value).shape),
        "dtype": str(np.asarray(value).dtype),
    }
    for key, value in observation.items()
}
observation_summary


In [ ]:
frame = observation["rgb"]

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(frame)
ax.set_title("Mock RGB observation")
ax.axis("off")
plt.show()

print(
    f"RGB range: {frame.min()}..{frame.max()} | "
    f"mean={frame.mean():.2f} | std={frame.std():.2f}"
)


## 3. Apply one controlled action

Instead of sampling a potentially destructive random action, this cell moves forward and rotates the camera by one slow increment. It then checks that exactly `FRAME_SKIP` ticks elapsed.


In [ ]:
action = noop_action()
action[1] = 1  # move forward
action[9] = 2  # yaw +6 degrees

tick_before = reset_info["tick"]
pose_before = observation["pose"].copy()

observation, reward, terminated, truncated, step_info = mock_env.step(action)

print(
    {
        "tick_before": tick_before,
        "tick_after": step_info["tick"],
        "reward": reward,
        "terminated": terminated,
        "truncated": truncated,
        "pose_changed": not np.array_equal(pose_before, observation["pose"]),
    }
)

assert step_info["tick"] - tick_before == FRAME_SKIP
assert not terminated
assert not truncated


## 4. Exercise: construct a valid action

Create an action that jumps, sprints, and turns left quickly. Predict which indices will be non-zero before running the answer scaffold.


In [ ]:
def exercise_action() -> np.ndarray:
    result = noop_action()
    result[2] = 1  # jump
    result[3] = 1  # sprint
    result[9] = 3  # yaw -18 degrees
    return result


candidate = exercise_action()
print("Action:", candidate.tolist())
print("Valid:", mock_env.action_space.contains(candidate))
assert mock_env.action_space.contains(candidate)


## 5. Optional real-Minecraft preflight

Leave `REAL_BACKEND = False` when running only the notebook test. To connect to Minecraft:

1. start `./gradlew runClient` in Terminal 1;
2. enter a single-player Survival world in the Minecraft window;
3. set `REAL_BACKEND = True` below;
4. run this and the following cells.

The preflight checks whether the local TCP bridge is listening before constructing the environment.


In [ ]:
REAL_BACKEND = False
BRIDGE_HOST = "127.0.0.1"
BRIDGE_PORT = 25570


def bridge_is_listening(host: str, port: int, timeout: float = 0.5) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


bridge_ready = bridge_is_listening(BRIDGE_HOST, BRIDGE_PORT)
print(f"Bridge listening: {bridge_ready}")

if REAL_BACKEND and not bridge_ready:
    raise RuntimeError(
        "The Fabric bridge is not reachable. Start Minecraft in Terminal 1, "
        "enter a single-player world, and retry."
    )
elif not REAL_BACKEND:
    print("Real backend disabled; continuing with the safe mock backend.")


In [ ]:
real_env = None

if REAL_BACKEND:
    real_env = gym.make(
        minecraft_gym.ENV_ID,
        backend="socket",
        host=BRIDGE_HOST,
        port=BRIDGE_PORT,
        width=WIDTH,
        height=HEIGHT,
        frame_skip=FRAME_SKIP,
        render_mode="rgb_array",
    )
    try:
        real_observation, real_reset_info = real_env.reset(seed=SEED)
    except Exception as exc:
        real_env.close()
        real_env = None
        raise RuntimeError(
            "The bridge answered, but reset failed. Confirm that Minecraft is "
            "inside a fully loaded single-player world, not at the title screen."
        ) from exc

    print("Connected to Minecraft.")
    print("Reset info:", real_reset_info)
    print("RGB shape:", real_observation["rgb"].shape)
else:
    print("Skipped real Minecraft connection.")


## 6. Run a short, safe episode

The cell uses five NOOP actions. With `REAL_BACKEND = True`, this advances Minecraft without intentionally moving, attacking, dropping items, or opening a GUI.


In [ ]:
test_env = real_env if REAL_BACKEND else mock_env
test_observation, test_reset_info = test_env.reset(seed=SEED)

ticks = [test_reset_info["tick"]]
rewards = []

for _ in range(5):
    test_observation, reward, terminated, truncated, info = test_env.step(
        noop_action()
    )
    ticks.append(info["tick"])
    rewards.append(reward)
    if terminated or truncated:
        break

print("Ticks:", ticks)
print("Rewards:", rewards)
assert all((right - left) == FRAME_SKIP for left, right in zip(ticks, ticks[1:]))


## 7. Cleanup and next steps

Always close the environment so injected keys are released and the integrated server is unfrozen. Next, try `scripts/record_human.py` to collect keyboard/mouse demonstrations, or replace the default reward with a custom `RewardFunction`.


In [ ]:
if real_env is not None:
    real_env.close()

mock_env.close()
print("Environments closed cleanly.")
